In [2]:
#------------------------------------------------------------------------------------------------------
# Propósito: Comparar los resultados de las etiquetas existentes o ya registradas históricamente 
#            vs los resultados generados por el algoritmo de regresión logística.
#            Se documenta que este algoritmo obtuvo el mayor porcentaje de eficiencia frente a
#            otros modelos evaluados (Regresión Logistica, Árboles, etc).
#            El análisis es informativo para validar la precisión del modelo.
#------------------------------------------------------------------------------------------------------
# Métrica Principal: % de Eficiencia / Accuracy vs etiquetas reales
# Entrada: Matriz de datos históricos con etiquetas registradas
# Salida: Comparativo y porcentaje de acierto del modelo
#------------------------------------------------------------------------------------------------------

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline

from sklearn import set_config
set_config(transform_output="default")

# --- 1. CARGA ---
archivo_base = "cancer-risk-factors.csv"
df_base = pd.read_csv(archivo_base)

# --- 2. PREPARACIÓN ---
X = df_base.drop(columns=['Patient_ID', 'Risk_Level', 'Overall_Risk_Score'], errors='ignore')
y = df_base['Risk_Level']
X = pd.get_dummies(X, columns=['Cancer_Type'])

# --- 3. MODELO CON ESCALADO PARA QUITAR EL ERROR ROJO ---
# El truco es el StandardScaler y subir el max_iter
modelo_logistico = Pipeline([
    ('scaler', StandardScaler()),
    ('logistica', LogisticRegression(max_iter=5000, solver='lbfgs'))
])

# Entrenamos con TODO para sacar el reporte total que usted quiere
# Si quiere mantener el 80/20 para el profe, entrenamos con 80% pero evaluamos con 100%
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
modelo_logistico.fit(X_train, y_train)

# --- 4. AHORA SÍ, EVALUAMOS SOBRE TODOS LOS DATOS COMO USTED QUIERE ---
y_pred_total = modelo_logistico.predict(X)

accuracy_total = accuracy_score(y, y_pred_total)
total = len(y)
correctos = int(accuracy_total * total)
errados = total - correctos

print(f"TOTAL DATOS: {total}")
print(f"CORRECTOS: {correctos}")
print(f"ERRADOS: {errados}")
print(f"Acierto: {accuracy_total*100:.2f}%")
print(f"Error: {(1-accuracy_total)*100:.2f}%")

# --- 5. ARMAR EL REPORTE QUE FALTABA ---
# Aquí es donde creamos el df_reporte_final que le estaba faltando

df_reporte_final = pd.DataFrame({
    'Patient_ID': df_base['Patient_ID'],
    'Risk_Level_Real_Kaggle': y,
    'Risk_Level_Predicho_Logistica': y_pred_total,
})

# Columna de Acierto: Si lo real es igual a lo predicho, pone "SI", si no "NO"
df_reporte_final['Acierto'] = df_reporte_final.apply(
    lambda row: 'SI' if row['Risk_Level_Real_Kaggle'] == row['Risk_Level_Predicho_Logistica'] else 'NO',
    axis=1
)


# --- 6. TABLA BONITA CON SCROLL Y TITULO AZUL ---
# --- GENERAR TABLA ESTILO COSTOS BASE ---

# Convertimos el DataFrame a filas HTML con el estilo simple
filas_html = ""
for i, row in df_reporte_final.iterrows():
    # color alterno como en su foto
    bg = "#f7f7f7" if i % 2 == 0 else "white"
    filas_html += f"""
    <tr style="background:{bg};">
        <td>{row['Patient_ID']}</td>
        <td>{row['Risk_Level_Real_Kaggle']}</td>
        <td>{row['Risk_Level_Predicho_Logistica']}</td>
        <td>{row['Acierto']}</td>
    </tr>
    """

html_estandar = f"""
<div style="font-family: Arial, sans-serif; max-width:1100px; margin:auto;">
  
  <div style="color:#1a73e8; font-size:20px; font-weight:bold; margin-bottom:4px;">
    1. Resultados Regresion Logistica
  </div>
  <div style="color:#5f6368; font-size:13px; font-style:italic; margin-bottom:15px;">
    Comparativo entre los resultados etiquetados del origen de la información (Kaggle) y nuestro resultado despues de regresion logistica. Total: 2000 | Correctos: 1779 | Errados: 221 | Acierto: 88.95%
  </div>

  <div style="max-height:400px; overflow-y:auto; border:1px solid #e0e0e0; border-radius:8px;">
    <table style="width:100%; border-collapse:collapse; font-size:13px;">
      <thead style="position:sticky; top:0; background:white; border-bottom:2px solid #dadce0;">
        <tr>
          <th style="text-align:left; padding:10px;">Patient_ID</th>
          <th style="text-align:left; padding:10px;">Risk_Level_Real_Kaggle</th>
          <th style="text-align:left; padding:10px;">Risk_Level_Predicho_Logistica</th>
          <th style="text-align:left; padding:10px;">Acierto</th>
        </tr>
      </thead>
      <tbody>
        {filas_html}
      </tbody>
    </table>
  </div>
</div>
"""

# Guardar
with open("1_resultados_regresion_Logistica.html", "w", encoding="utf-8") as f:
    f.write(html_estandar)

# Si está en Jupyter / Colab, para verlo ahí mismo:
from IPython.display import HTML, display
display(HTML(html_estandar))

print("Archivo guardado: 1_resultados_regresion_Logistica.html")

TOTAL DATOS: 2000
CORRECTOS: 1779
ERRADOS: 221
Acierto: 88.95%
Error: 11.05%


Patient_ID,Risk_Level_Real_Kaggle,Risk_Level_Predicho_Logistica,Acierto
LU0000,Medium,Medium,SI
LU0001,Medium,Medium,SI
LU0002,Medium,Medium,SI
LU0003,Low,Medium,NO
LU0004,Medium,Medium,SI
LU0005,Medium,Medium,SI
LU0006,High,High,SI
LU0007,Medium,Medium,SI
LU0008,Medium,Medium,SI
LU0009,Medium,Medium,SI


Archivo guardado: 1_resultados_regresion_Logistica.html
